# Neural Network (MLP)

Train and tune a Keras MLP for flight cancellation prediction. We try several architectures and pick the best one based on validation PR-AUC, then tune the decision threshold for F2.

Both feature versions (with and without `IS_COVID`) are tested.

1. Load the preprocessed data
2. Define the configs we want to try
3. Train each config with early stopping on val PR-AUC
4. Pick the best, tune threshold for F2
5. Report final test metrics and save the model

In [1]:
%pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.4/572.4 MB 39.8 MB/s  0:00:15m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 27.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 26.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 32.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 31.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [tensorflow]9 [tensorflow]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, metrics
from sklearn.metrics import average_precision_score, fbeta_score

MODEL_DIR = 'model_weights'
os.makedirs(MODEL_DIR, exist_ok=True)

I0000 00:00:1774501353.935289 3291606 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Load the data

Both feature versions from `artifacts/`. Targets are shared.

In [3]:
datasets = {}
for version in ['with_covid', 'no_covid']:
    datasets[version] = {
        'X_train': pd.read_parquet(f'artifacts/X_train_{version}.parquet'),
        'X_val':   pd.read_parquet(f'artifacts/X_val_{version}.parquet'),
        'X_test':  pd.read_parquet(f'artifacts/X_test_{version}.parquet'),
    }

y_train = pd.read_parquet('artifacts/y_train.parquet')['CANCELLED'].values
y_val   = pd.read_parquet('artifacts/y_val.parquet')['CANCELLED'].values
y_test  = pd.read_parquet('artifacts/y_test.parquet')['CANCELLED'].values

# compute class weight from training labels
n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
keras_class_weight = {0: 1.0, 1: n_neg / n_pos}

print(f'Train: {len(y_train):,} rows  |  cancel rate: {y_train.mean():.2%}')
print(f'Val:   {len(y_val):,} rows  |  cancel rate: {y_val.mean():.2%}')
print(f'Test:  {len(y_test):,} rows  |  cancel rate: {y_test.mean():.2%}')
print(f'Class weight ratio: {keras_class_weight[1]:.1f}')
print(f'\nWith COVID: {datasets["with_covid"]["X_train"].shape[1]} features')
print(f'No COVID:   {datasets["no_covid"]["X_train"].shape[1]} features')

Train: 2,186,803 rows  |  cancel rate: 2.92%
Val:   349,591 rows  |  cancel rate: 2.16%
Test:  698,587 rows  |  cancel rate: 2.06%
Class weight ratio: 33.3

With COVID: 25 features
No COVID:   24 features


## Model configurations

We define a set of architectures to try. Each config specifies the hidden layer sizes, learning rate, batch size, and any extras (LR schedule, weight decay). All configs use BatchNorm, Dropout, binary crossentropy, and early stopping on val PR-AUC.

The configs go from simple to complex — if a small network already works well, there's no need for a massive one.

In [4]:
configs = [
    {'name': 'MLP [64,32] lr=1e-3 bs=1024',
     'hidden': [64, 32], 'lr': 0.001, 'batch': 1024, 'epochs': 30,
     'dropout': 0.3, 'schedule': False, 'weight_decay': None},

    {'name': 'MLP [128,64] lr=1e-3 bs=256',
     'hidden': [128, 64], 'lr': 0.001, 'batch': 256, 'epochs': 30,
     'dropout': 0.3, 'schedule': False, 'weight_decay': None},

    {'name': 'MLP [128,64] lr=1e-4 bs=256',
     'hidden': [128, 64], 'lr': 0.0001, 'batch': 256, 'epochs': 50,
     'dropout': 0.3, 'schedule': False, 'weight_decay': None},

    {'name': 'MLP [256,128,64] lr=1e-4',
     'hidden': [256, 128, 64], 'lr': 0.0001, 'batch': 256, 'epochs': 50,
     'dropout': 0.3, 'schedule': False, 'weight_decay': None},

    {'name': 'MLP [256,128,64] LR schedule',
     'hidden': [256, 128, 64], 'lr': 0.001, 'batch': 256, 'epochs': 50,
     'dropout': 0.3, 'schedule': True, 'weight_decay': None},

    {'name': 'MLP [256,128,64] LR sched + WD',
     'hidden': [256, 128, 64], 'lr': 0.001, 'batch': 256, 'epochs': 50,
     'dropout': 0.3, 'schedule': True, 'weight_decay': 1e-4},

    {'name': 'MLP [512,256,128] LR sched + WD',
     'hidden': [512, 256, 128], 'lr': 0.001, 'batch': 256, 'epochs': 50,
     'dropout': 0.3, 'schedule': True, 'weight_decay': 1e-4},

    {'name': 'MLP [256,128] lr=1e-3 bs=256',
     'hidden': [256, 128], 'lr': 0.001, 'batch': 256, 'epochs': 50,
     'dropout': 0.3, 'schedule': False, 'weight_decay': None},
]

print(f'{len(configs)} configs defined')

8 configs defined


## Helpers

In [5]:
def build_model(n_features, hidden_sizes, lr, dropout, use_schedule, weight_decay):
    """Build and compile a Keras MLP from a config."""
    model = keras.Sequential()
    for i, units in enumerate(hidden_sizes):
        if i == 0:
            model.add(layers.Dense(units, activation='relu', input_shape=(n_features,)))
        else:
            model.add(layers.Dense(units, activation='relu'))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation='sigmoid'))

    if use_schedule:
        lr_used = keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=lr, decay_steps=1000, decay_rate=0.9)
    else:
        lr_used = lr

    if weight_decay:
        opt = keras.optimizers.AdamW(learning_rate=lr_used, weight_decay=weight_decay)
    else:
        opt = keras.optimizers.Adam(learning_rate=lr_used)

    model.compile(
        optimizer=opt, loss='binary_crossentropy',
        metrics=[metrics.AUC(curve='PR', name='pr_auc')])
    return model


def find_best_threshold(y_true, y_proba, beta=2):
    best_t, best_score = 0.5, 0.0
    for t in np.linspace(0, 1, 101):
        score = fbeta_score(y_true, (y_proba >= t).astype(int), beta=beta, zero_division=0)
        if score > best_score:
            best_score, best_t = score, t
    return best_t, best_score

## Train all configs

For each feature version, we train every config, track val PR-AUC, and keep the best one. Then we threshold-tune and evaluate on test.

This takes a while — 8 configs x 2 versions = 16 training runs.

In [6]:
all_results = []

for version in ['with_covid', 'no_covid']:
    X_tr = datasets[version]['X_train']
    X_v  = datasets[version]['X_val']
    X_te = datasets[version]['X_test']
    n_features = X_tr.shape[1]

    print(f'\n{"=" * 60}')
    print(f'  {version}  ({n_features} features)')
    print(f'{"=" * 60}')

    best_prauc = -1
    best_model = None
    best_config_name = None
    config_results = []

    for cfg in configs:
        print(f'\n  Training: {cfg["name"]}')
        tf.random.set_seed(42)

        model = build_model(
            n_features, cfg['hidden'], cfg['lr'],
            cfg['dropout'], cfg['schedule'], cfg['weight_decay'])

        es = callbacks.EarlyStopping(
            monitor='val_pr_auc', mode='max',
            patience=5, restore_best_weights=True, verbose=0)

        history = model.fit(
            X_tr.values, y_train,
            validation_data=(X_v.values, y_val),
            epochs=cfg['epochs'], batch_size=cfg['batch'],
            class_weight=keras_class_weight,
            callbacks=[es], verbose=0)

        val_prauc = max(history.history['val_pr_auc'])
        stopped = len(history.history['val_pr_auc'])
        print(f'    val PR-AUC: {val_prauc:.4f}  (stopped at epoch {stopped})')

        config_results.append({'config': cfg['name'], 'val_pr_auc': val_prauc})

        if val_prauc > best_prauc:
            best_prauc = val_prauc
            best_model = model
            best_config_name = cfg['name']

    # show all configs ranked
    print(f'\n  Config ranking ({version}):')
    for r in sorted(config_results, key=lambda x: x['val_pr_auc'], reverse=True):
        marker = ' <-- best' if r['config'] == best_config_name else ''
        print(f'    {r["val_pr_auc"]:.4f}  {r["config"]}{marker}')

    # threshold tune on val
    val_proba = best_model.predict(X_v.values, verbose=0).ravel()
    best_t, val_f2 = find_best_threshold(y_val, val_proba)
    print(f'\n  Best threshold: {best_t:.2f} (val F2 = {val_f2:.4f})')

    # test evaluation
    test_proba = best_model.predict(X_te.values, verbose=0).ravel()
    test_preds = (test_proba >= best_t).astype(int)
    test_prauc = average_precision_score(y_test, test_proba)
    test_f2 = fbeta_score(y_test, test_preds, beta=2, zero_division=0)

    print(f'  Test PR-AUC: {test_prauc:.4f}  |  Test F2: {test_f2:.4f}')

    all_results.append({
        'Version': version,
        'Best Config': best_config_name,
        'Threshold': best_t,
        'PR-AUC': test_prauc,
        'F2': test_f2,
    })

    # save best model
    out_path = f'{MODEL_DIR}/nn_{version}.keras'
    best_model.save(out_path)
    print(f'  Saved: {out_path}')


  with_covid  (25 features)

  Training: MLP [64,32] lr=1e-3 bs=1024


/common/home/users/c/cl.zhao.2025/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1774501358.382767 3291606 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22327 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1774501361.654744 3291872 service.cc:153] XLA service 0x14f584033620 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774501361.654769 3291872 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3090, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.6.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1774501361.7

    val PR-AUC: 0.1279  (stopped at epoch 9)

  Training: MLP [128,64] lr=1e-3 bs=256


I0000 00:00:1774501408.468059 3291869 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_97145__.28
I0000 00:00:1774501408.547663 3291869 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1774501409.812518 3291869 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1774501410.698851 3292553 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_20', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1774501410.957328 3291869 dot_search_space.cc:240] All configs were filtered out because none

    val PR-AUC: 0.1370  (stopped at epoch 11)

  Training: MLP [128,64] lr=1e-4 bs=256


...

    val PR-AUC: 0.1376  (stopped at epoch 9)

  Training: MLP [512,256,128] LR sched + WD


I0000 00:00:1774504301.029775 3291872 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8552422__.36
I0000 00:00:1774504316.871902 3291870 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8552422__.36


    val PR-AUC: 0.1402  (stopped at epoch 11)

  Training: MLP [256,128] lr=1e-3 bs=256


I0000 00:00:1774504463.882577 3291869 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8994302__.28
I0000 00:00:1774504477.363718 3291869 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8994302__.28


    val PR-AUC: 0.1436  (stopped at epoch 14)

  Config ranking (no_covid):
    0.1436  MLP [256,128] lr=1e-3 bs=256 <-- best
    0.1424  MLP [256,128,64] lr=1e-4
    0.1402  MLP [512,256,128] LR sched + WD
    0.1376  MLP [256,128,64] LR sched + WD
    0.1373  MLP [128,64] lr=1e-3 bs=256
    0.1368  MLP [128,64] lr=1e-4 bs=256
    0.1325  MLP [64,32] lr=1e-3 bs=1024
    0.1305  MLP [256,128,64] LR schedule

  Best threshold: 0.56 (val F2 = 0.2491)
  Test PR-AUC: 0.0514  |  Test F2: 0.1407
  Saved: model_weights/nn_no_covid.keras


## Results

In [7]:
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))

   Version                  Best Config  Threshold   PR-AUC       F2
with_covid MLP [256,128] lr=1e-3 bs=256       0.63 0.077677 0.191942
  no_covid MLP [256,128] lr=1e-3 bs=256       0.56 0.051370 0.140659
